# Adaptive Graph Construction for GeoContrastNet
## Google Colab Execution Notebook

This notebook runs the full pipeline for comparing KNN baseline vs adaptive graph construction.

**Estimated runtime on GPU (T4):** ~45 minutes

**Before running:** Replace `GITHUB_REPO` in CELL 1 with your actual GitHub repository URL.

# CELL 1: Mount Google Drive and Clone Repository from GitHub

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/gdrive')

import os
os.chdir('/content')

# Clone GeoContrastNet repo from GitHub
# IMPORTANT: Replace URL with your actual GitHub repo
GITHUB_REPO = "https://github.com/yourusername/GeoContrastNet.git"

print("=" * 70)
print("STEP 1: Clone GeoContrastNet from GitHub")
print("=" * 70)
print(f"Repository URL: {GITHUB_REPO}")
print("(If repo is private, use: https://<TOKEN>@github.com/<user>/GeoContrastNet.git)")
print()

import subprocess
result = subprocess.run(["git", "clone", GITHUB_REPO], cwd="/content")

# Verify clone
import time
time.sleep(1)
if os.path.exists('/content/GeoContrastNet'):
    print("\n[SUCCESS] GeoContrastNet cloned to /content/")
    os.chdir('/content/GeoContrastNet')
    print("\nRepository contents:")
    !ls -la | head -20
else:
    print("\n[ERROR] Clone failed. Possible reasons:")
    print("  1. Check GitHub repo URL is correct")
    print("  2. For private repos, use Personal Access Token: https://<TOKEN>@github.com/...")
    print("  3. Make sure git is installed")
    raise RuntimeError("Failed to clone repository")

# CELL 2: Install Dependencies

In [ ]:
print("\n" + "="*70)
print("STEP 2: Installing dependencies")
print("="*70 + "\n")

!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q dgl scikit-learn pillow pyyaml matplotlib tqdm requests pandas seaborn
!python -m spacy download en_core_web_lg

print("\nDependencies installed!")

# Verify
import torch
import dgl
import numpy as np
print(f"PyTorch: {torch.__version__}")
print(f"DGL: {dgl.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# CELL 3: Download FUNSD Dataset

In [ ]:
print("\n" + "="*70)
print("STEP 3: Download FUNSD Dataset")
print("="*70 + "\n")

import os
data_dir = '/content/GeoContrastNet/src/data/datasets/FUNSD'
if not os.path.exists(data_dir):
    print("FUNSD not found, downloading...")
    os.chdir('/content/GeoContrastNet/src/data')
    !python download.py
    os.chdir('/content/GeoContrastNet')
    print("\nFUNSD download complete!")
else:
    print(f"FUNSD already exists at {data_dir}")

print("\nFUNSD contents:")
!ls -la src/data/datasets/FUNSD/ | head -20

# CELL 4: Build Both Graph Variants (Stage 1)
## KNN Baseline and Adaptive

In [ ]:
import subprocess
import sys

print("\n" + "="*70)
print("STAGE 1: Building KNN and Adaptive graphs")
print("="*70 + "\n")
print("This may take 5-10 minutes on CPU, 1-2 minutes on GPU...\n")

result = subprocess.run([sys.executable, 'scripts/build_baseline_vs_adaptive.py'],
                       cwd='/content/GeoContrastNet')

if result.returncode == 0:
    print("\n[SUCCESS] Graph building complete!")
    !ls -la /content/GeoContrastNet/graphs_stage1/adaptive_experiment/
else:
    print("\n[ERROR] Graph building failed!")
    sys.exit(1)

# CELL 5: Visualize Graphs
## KNN vs Adaptive Side-by-Side

In [ ]:
print("\n" + "="*70)
print("Visualizing KNN vs Adaptive graphs")
print("="*70 + "\n")

result = subprocess.run([sys.executable, 'scripts/visualize_graphs.py'],
                       cwd='/content/GeoContrastNet')

print("\n[SUCCESS] Visualizations saved to outputs/graph_compare/")
!ls -la /content/GeoContrastNet/outputs/graph_compare/ | head -20

# CELL 6: Train Stage 2 on KNN Baseline
## 100 Epochs

In [ ]:
print("\n" + "="*70)
print("STAGE 2a: Training on KNN baseline graphs (100 epochs)")
print("Estimated time: 15-25 min on GPU, 2-4 hours on CPU")
print("="*70 + "\n")

result = subprocess.run([sys.executable, 'main.py', '--run-name', 'run111_knn'],
                       cwd='/content/GeoContrastNet')

if result.returncode == 0:
    print("\n[SUCCESS] KNN training complete!")
    !ls -la /content/GeoContrastNet/runs/run111_knn/
else:
    print("\n[ERROR] KNN training failed!")

# CELL 7: Train Stage 2 on Adaptive Graphs
## 100 Epochs

In [ ]:
print("\n" + "="*70)
print("STAGE 2b: Training on adaptive graphs (100 epochs)")
print("Estimated time: 15-25 min on GPU, 2-4 hours on CPU")
print("="*70 + "\n")

result = subprocess.run([sys.executable, 'main.py', '--run-name', 'run111_adaptive'],
                       cwd='/content/GeoContrastNet')

if result.returncode == 0:
    print("\n[SUCCESS] Adaptive training complete!")
    !ls -la /content/GeoContrastNet/runs/run111_adaptive/
else:
    print("\n[ERROR] Adaptive training failed!")

# CELL 8: Collect and Compare Results

In [ ]:
print("\n" + "="*70)
print("Comparing KNN vs Adaptive results")
print("="*70 + "\n")

result = subprocess.run([sys.executable, 'scripts/collect_results.py'],
                       cwd='/content/GeoContrastNet')

print("\n[SUCCESS] Results aggregated!")

# Display comparison
import pandas as pd
from pathlib import Path

comparison_csv = Path('/content/GeoContrastNet/outputs/comparison.csv')
if comparison_csv.exists():
    df = pd.read_csv(comparison_csv)
    print("\n" + "="*70)
    print("COMPARISON RESULTS")
    print("="*70)
    print(df.to_string())
    print("\nSave this table for your thesis!")

# CELL 9: Save Results to Google Drive
## All outputs → /gdrive/My Drive/thesis2/

In [ ]:
print("\n" + "="*70)
print("Saving results to Google Drive")
print("="*70)

import os

# Ensure thesis2 folder exists
!mkdir -p /gdrive/'My Drive'/thesis2

# Copy the full GeoContrastNet folder with results back to Drive
print("\nCopying runs/ and outputs/ to Google Drive...")
!cp -r /content/GeoContrastNet/runs "/gdrive/My Drive/thesis2/" 2>/dev/null && echo "OK: Copied runs/" || echo "WARN: Could not copy runs"
!cp -r /content/GeoContrastNet/outputs "/gdrive/My Drive/thesis2/" 2>/dev/null && echo "OK: Copied outputs/" || echo "WARN: Could not copy outputs"
!cp -r /content/GeoContrastNet/graphs_stage1 "/gdrive/My Drive/thesis2/" 2>/dev/null && echo "OK: Copied graphs_stage1/" || echo "WARN: Could not copy graphs"

# Also create a zip for easy download
print("\nCreating backup zip file...")
!cd /content/GeoContrastNet && zip -r "/gdrive/My Drive/thesis2/GeoContrastNet_results_backup.zip" runs outputs graphs_stage1 2>/dev/null

print("\nAll results saved to /gdrive/My Drive/thesis2/")
print("Files available:")
!ls -lh "/gdrive/My Drive/thesis2/" | head -20

# Summary: Pipeline Complete!

## Results Saved to Google Drive
**Location:** `/gdrive/My Drive/thesis2/`

### Files Generated:
- `runs/run111_knn/` — **KNN baseline results**
- `runs/run111_adaptive/` — **Adaptive results**
- `outputs/comparison.csv` — **SHOW TO SUPERVISOR** ← Side-by-side metrics
- `outputs/comparison.json` — Detailed metrics in JSON
- `outputs/graph_compare/` — **Visual proof** ← 4-panel comparisons
- `graphs_stage1/adaptive_experiment/` — Graph binary files
- `GeoContrastNet_results_backup.zip` — Backup archive

### Next Steps:
1. ✅ Download `comparison.csv` from Google Drive
2. ✅ Review metrics: KNN vs Adaptive F1 score
3. ✅ Check `graph_compare/*.png` visualizations
4. ✅ If Adaptive ≥ KNN, document as thesis contribution
5. ✅ Show supervisor the comparison.csv + visualizations

### For Full Documentation:
See `ADAPTIVE_GRAPH_CONSTRUCTION.md` and `IMPLEMENTATION_SUMMARY.md` in the repo